In [25]:
import os
import csv
from typing import Literal
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from tavily import TavilyClient
from deepagents import create_deep_agent

load_dotenv()
tavily_api_key = os.getenv("TAVILY_API_KEY")
tavily = TavilyClient(api_key=tavily_api_key)
openrouter_api = os.getenv("OPENROUTER_API_KEY")



def internet_search(
    query: str
):
    """
    Use this tool to make an internet search by providing a query.
    """
    print("PERFORMING A SEARCH ON: "+query)

    return tavily.search(query, max_results=5, topic="general")


def list_subdirectory_filenames(subdirectory: str= "files", recursive: bool = True):
    """
    Use this tool to list filenames in a given subdirectory. By default the subdirectory files is used.
    """

    base_dir = os.getcwd()
    target_dir = subdirectory if os.path.isabs(subdirectory) else os.path.join(base_dir, subdirectory)

    if not os.path.isdir(target_dir):
        return {
            "error": f"Directory not found: {target_dir}",
            "files": [],
        }

    files: list[str] = []

    if recursive:
        for root, _, filenames in os.walk(target_dir):
            for filename in filenames:
                full_path = os.path.join(root, filename)
                files.append(os.path.relpath(full_path, target_dir))
    else:
        for name in os.listdir(target_dir):
            full_path = os.path.join(target_dir, name)
            if os.path.isfile(full_path):
                files.append(name)

    files.sort()
    print("FILES IM UNTERVERZEICHNIS SIND:"+str(files))
    return {
        "directory": target_dir,
        "recursive": recursive,
        "file_count": len(files),
        "files": files,
    }


def read_target_csv_file(filename: str, max_rows: int = 50):
    """
    Use this tool to open a CSV file and return its values in a readable structured format.
    Provide a relative or absolute filename to the CSV.
    """
    print("FILE WIRD AUSGELESEN:"+filename)
    base_dir = os.getcwd()
    target_file = filename if os.path.isabs(filename) else os.path.join(base_dir, filename)

    if not os.path.isfile(target_file):
        return {
            "error": f"CSV file not found: {target_file}",
            "columns": [],
            "rows": [],
        }

    rows: list[dict[str, str]] = []
    with open(target_file, mode="r", encoding="utf-8-sig", newline="") as csv_file:
        reader = csv.DictReader(csv_file)
        columns = reader.fieldnames or []

        for i, row in enumerate(reader):
            if i >= max_rows:
                break
            rows.append({k: (v if v is not None else "") for k, v in row.items()})

    return {
        "filename": target_file,
        "columns": columns,
        "row_count_returned": len(rows),
        "max_rows": max_rows,
        "rows": rows,
    }




llm = ChatOpenRouter(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=openrouter_api 
)


agent = create_deep_agent(
    model=llm,
    system_prompt="""You are a generic Deep Agent, an expert orchestrator designed to perform any task.
Your primary goal is to use the provided skill library to handle specialized requirements on-demand.
...""",
    tools=[internet_search, list_subdirectory_filenames, read_target_csv_file]
)



In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "Perform an internet search and find out who the current mayor of the city of Berne is"}]})
print(result["messages"][-1].content)

PERFORMING A SEARCH ON: current mayor of Bern Switzerland
The current mayor of Bern (Berne) is **Marieke Kruit**, who has held the office since 2025. She is a member of the Social Democratic Party (SPS/PSS) and is the first woman to serve as mayor of the city.


In [27]:
result = agent.invoke({"messages": [{"role": "user", "content": "Suche die File im Unterverzeichnis files zu Energie im Kanton Graubünden und lies dann darin aus wieviel Photovoltaik im 2024 generiert wurde."}]})
print(result["messages"][-1].content)

TooManyRequestsResponseError: Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day